# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the **FAIR²** dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will learn to load the dataset, review its structure via Croissant `@id` references, extract Tables by their record set IDs, and perform exploratory data analysis (EDA) and visualization.

### Dataset Source
The dataset is described by its Croissant schema, referenced here:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Install the mlcroissant library if not present
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset overview
print(f"Title: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")


## 2. Data Overview
Let's review the available record sets, fields, and their `@id` values. In Croissant, a record set is a table-like collection, and each field and column is identified by a unique `@id`. Referencing data by `@id` ensures reproducible and portable workflows.

In [ ]:
# List all record sets and their fields using their @id

def get_record_sets(dataset):
    """Return a list of (record_set_id, name, fields)"""
    results = []
    for rs in dataset.record_sets:
        rs_id = rs['@id'] if '@id' in rs else None
        rs_name = rs.get('name', '(Unnamed)')
        # List fields under 'field'
        field_objs = rs.get('field', [])
        if not isinstance(field_objs, list):
            field_objs = [field_objs]
        fields = []
        for field in field_objs:
            # if the field is a reference, it's a string @id
            field_id = field if isinstance(field, str) else field.get('@id', None)
            if field_id:
                fields.append(field_id)
        results.append((rs_id, rs_name, fields))
    return results

record_sets_info = get_record_sets(dataset)

print("Available record sets and fields (by @id):\n")
for rs_id, rs_name, fields in record_sets_info:
    print(f" - RecordSet: {rs_name} (@id: {rs_id})")
    if fields:
        for f in fields:
            print(f"     - Field @id: {f}")
    else:
        print("     (No fields listed)")
if not record_sets_info:
    print("No record sets were found in the metadata.\nIf so, please re-load dataset and check the schema content.")

## 3. Data Extraction
Let's extract the data for all available record sets using their `@id`. We will load each table into a Pandas DataFrame. Use the list of record set `@id`s from the previous step. For demonstration, we'll preview columns for the first record set (if available).

In [ ]:
# Gather all record set @id values from previous section
record_set_ids = [rs_id for rs_id, rs_name, fields in record_sets_info if rs_id]
dataframes = {}

if not record_set_ids:
    print('No record sets found to extract data.')
else:
    # Extract data from each record set by @id
    for record_set_id in record_set_ids:
        print(f"Extracting records for RecordSet @id: {record_set_id} ...")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  - Loaded {len(df)} rows, columns: {list(df.columns)}\n")
        except Exception as e:
            print(f"  - Error loading record set {record_set_id}: {e}\n")
    # As an example, preview the first DataFrame
    if dataframes:
        first_rs_id = list(dataframes.keys())[0]
        print(f"Head of first record set (@id: {first_rs_id}):")
        display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply common EDA steps using the extracted DataFrame. We'll demonstrate filtering, normalization, and grouping. 

> **Note**: You must reference columns by their full Croissant `@id` (or mapped DataFrame column, depending on loader behavior). If you get a `KeyError`, check the exact column names using `df.columns`.

- We'll look for a numeric field and a grouping field in the first record set for demonstration.

In [ ]:
# Choose a record set and numeric/grouping fields by @id (based on previous outputs)

import numpy as np

if not dataframes:
    print("No dataframes available for analysis.")
else:
    # Pick the first record set as an example
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()

    print(f"Columns for record set @id '{record_set_id}':\n{df.columns.tolist()}\n")

    # Try to pick a numeric column automatically, fallback to user (assume 'log_likelihood' for demonstration)
    numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]
    if not numeric_candidates:
        # try by partial matches
        numeric_candidates = [c for c in df.columns if 'log' in c.lower() or 'numeric' in c.lower() or 'score' in c.lower()]

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field for EDA: {numeric_field}\n")
    else:
        numeric_field = df.columns[0]
        print(f"No obvious numeric field found. Using first column: {numeric_field}")

    # Set filter threshold for demonstration
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
    if threshold is not None:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows.")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
        )
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by categorical column (e.g., any column with 'group', 'ward', or 'region' in name)
        group_candidates = [c for c in df.columns if any(x in c.lower() for x in ['group', 'ward', 'region', 'category', 'gender'])]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df)
        else:
            print("\nNo clear categorical field found for grouping.")
    else:
        print(f"Numeric filter not applied; field '{numeric_field}' is not numeric.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if grouping is available, compare group means. This enhances understanding of the data's structure and highlights important patterns.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No dataframes to plot.")
else:
    # Use previous variables if defined
    try:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=20, color='skyblue')
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        # Barplot if grouping exists
        if 'group_field' in locals():
            plt.figure(figsize=(10,4))
            sns.barplot(data=grouped_df, x=group_field, y=numeric_field, palette='Set2')
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.xticks(rotation=35)
            plt.show()
    except Exception as e:
        print(f"Plotting failed: {e}")

## 6. Conclusion
With the `mlcroissant` library, we loaded a FAIR-compliant dataset using Croissant `@id` references, explored its record sets and fields, and performed simple EDA and visualization in a fully reproducible way.

**Key findings and next steps:**
- The dataset structure is clearly described with Croissant `@id` for all tables and columns.
- Numeric analysis and filtering can be automated by discovering field types.
- This workflow enables rapid FAIR dataset exploration, and can be extended for feature engineering, advanced modeling, or public dissemination.

For more advanced EDA, consider integrating profiling libraries such as [pandas-profiling](https://github.com/pandas-profiling/pandas-profiling), or connecting to ML pipelines.
